# 09 — Beam Steering and Adaptive Nulling


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinculum3141-ship-it/active-radar-tracker-basics/blob/radar-tracker-notebooks/beginner/09-beam-steering-adaptive-nulling.ipynb)


## What this notebook teaches

Notebook 08 scanned the array to *find* a target's direction and showed that a Capon scan can survive a strong interferer. This notebook makes that idea concrete and complete: you will *point* the array at the target with steering weights, and then force a *deep null* on the interferer with linear-constrained minimum-variance (LCMV) weights. Finally you will see the payoff where it counts - the interferer's echo disappears from the range-Doppler map after cancellation, while the target's echo stays.

By the end of this notebook, you should be able to explain:

- what the array weights do and how steering points the main lobe at the target,
- how an LCMV constraint forces a null at the interferer's angle,
- how deep that null is, and why steering alone cannot provide it,
- why the null is applied *before* the matched filter,
- and how an adaptive null removes the interferer from the range-Doppler map while preserving the target.

Keep these five questions in mind as you work through the cells. A dedicated section at the end answers each one directly.


## Setup and baseline values

We reuse the array from Notebooks 07 and 08: 8 elements at half-wavelength spacing, a target at +20 degrees, and a strong interferer at -30 degrees. The scene also carries range and Doppler so we can watch the interference in a range-Doppler map: the target at 1000 m moving at +8 m/s, and the interferer at the same range moving at -8 m/s, 30 dB stronger.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vinculum3141-ship-it/active-radar-tracker-basics.git"
BRANCH_NAME = "radar-tracker-notebooks"
REPO_DIR = Path("/content/active-radar-tracker-basics")

in_colab = "google.colab" in sys.modules

if in_colab and not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH_NAME, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

if in_colab:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

    repo_root = os.path.abspath(".")
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

    print(f"Ready in Colab from {REPO_DIR}")
else:
    repo_root = os.path.dirname(os.getcwd())
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print("Running locally; the repository is already available in this workspace.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beginner.helpers import baseline_spec
from beginner.helpers.array import steering_vector
from beginner.helpers.math import wavelength_m, range_from_delay_samples, delay_samples_for_range
from beginner.helpers.waveforms import lfm_chirp, matched_filter
from beginner.helpers.doppler import build_pulse_stack, range_doppler_map
from beginner.helpers.steering import steering_weights, lcmv_weights
from beginner.helpers.plotting import apply_notebook_style

np.random.seed(42)
apply_notebook_style()
radar_spec = baseline_spec()

lam = wavelength_m(radar_spec.fc_hz)
n_elements = 8
d_spacing = lam / 2.0
angles = np.linspace(-90, 90, 1801)

print(f"8 elements, half-wavelength spacing, lambda = {lam*100:.2f} cm")
print(f"Target at +{radar_spec.target_angle_deg:.0f} deg, interferer at {radar_spec.interferer_angle_deg:.0f} deg")

## Where we are in the story

After Steering and the beam pattern, you know the array can be *pointed*. After DOA, you know it can also be *scanned* and that an adaptive scan survives interference. This notebook pulls it together: instead of just pointing or scanning, you will choose a *weight vector* - the set of complex gains applied to the elements - that both points at the target and cancels the interferer. The rest of this notebook is about how those weights are chosen and what they do to a real range-Doppler map.


## Combining the array is choosing weights

Every way of using the array boils down to one weight vector w: one complex number per element. The combined output is

    y = w^H x

where x is the snapshot at one moment and w^H is the conjugate transpose. The beam pattern you drew in Notebook 07 is just |w^H a(theta)|^2 - how strongly this weighting responds to a wave from each angle theta. So *pointing* the array and *nulling* the interferer are both the same act: picking w.


## Steering the beam at the target

To listen to the target at 20 degrees, choose weights w = a(20)/N - the target's own steering vector, scaled to unit response. These weights line up the elements so a wave from 20 degrees adds coherently (the main lobe points there) and a wave from elsewhere falls apart (the sidelobes are much weaker).


In [ ]:
w_steer = steering_weights(np.radians(20.0), n_elements, d_spacing, lam)

def response_db(w):
    P = np.array([abs(w.conj() @ steering_vector(np.radians(a), n_elements, d_spacing, lam))**2 for a in angles])
    return 10.0 * np.log10(P / P.max())

res_steer = response_db(w_steer)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(angles, res_steer, color="#1b9e77", linewidth=2)
ax.axvline(20, color="#d95f02", linestyle=":", label="target +20 deg")
ax.axvline(-30, color="#7570b3", linestyle=":", label="interferer -30 deg")
ax.set_xlabel("Angle (deg)")
ax.set_ylabel("Response (dB)")
ax.set_title("Steering weights: the main lobe points at +20 deg")
ax.legend()
ax.set_ylim(-40, 2)
plt.tight_layout()
plt.show()

print(f"Main lobe peaks at {angles[np.argmax(res_steer)]:.0f} deg (the target)")
print(f"Response at the interferer (-30 deg): {res_steer[np.argmin(np.abs(angles+30))]:.1f} dB (still leaks through a sidelobe)")

## Steering alone cannot reject the interferer

Steering points the beam correctly, but the beam still has sidelobes. At -30 degrees the response is only about -18.6 dB below the main lobe, and our interferer is 30 dB *stronger* than the target. The interferer therefore arrives at the output about 30 - 18.6 = 11 dB above the target even though we steered straight at the target. Steering points, but it does not reject - it has no way to demand a null at the interferer.


## Forcing a null: the LCMV constraint

We want weights that fix two things at once: unit response on the target *and* zero response on the interferer. Write the steering vectors of the target and interferer as the columns of a constraint matrix C = [a(20), a(-30)]. The requirements are

    C^H w = [1, 0]

meaning "respond with gain 1 to the target and gain 0 to the interferer". Among the many weights that satisfy this, the notebook uses the minimum-norm solution, which is the minimum-variance solution for spatially white noise:

    w = C (C^H C)^{-1} f,   f = [1, 0]

This is a constraint-driven way to get the same nulling idea you met with Capon, but now you state the null angle directly instead of scanning for it. A general LCMV design for coloured noise also uses the noise covariance; the expression above is the white-noise case.


In [ ]:
C, f, w_lcmv = lcmv_weights(np.radians(20.0), [-np.radians(30.0)], n_elements, d_spacing, lam)

print("Constraint matrix C, one steering vector per column:")
print(np.round(C, 3))
print(f"\nDesired response f = {f}")
print(f"C^H w = {np.round(C.conj().T @ w_lcmv, 4)}  (should be [1, 0])")

res_lcmv = response_db(w_lcmv)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(angles, res_lcmv, color="#7570b3", linewidth=2)
ax.axvline(20, color="#d95f02", linestyle=":", label="target +20 deg")
ax.axvline(-30, color="#7570b3", linestyle=":", label="interferer -30 deg")
ax.set_xlabel("Angle (deg)")
ax.set_ylabel("Response (dB)")
ax.set_title("LCMV weights: null at -30 deg, target kept at 0 dB")
ax.legend()
ax.set_ylim(-40, 2)
plt.tight_layout()
plt.show()

print(f"Response at the target (+20 deg)  : {res_lcmv[np.argmin(np.abs(angles-20))]:.1f} dB")
print(f"Response at the interferer (-30): {res_lcmv[np.argmin(np.abs(angles+30))]:.1f} dB (deep null)")

## Before and after: the beam pattern

Overlaying the two response curves makes the change plain. Steering alone leaves a -18.6 dB sidelobe at the interferer; the constrained weights carve a much deeper null. The computed value near -319 dB is limited by floating-point precision, not a promise of real-world rejection. The target keeps full, 0 dB response in both. In this ideal array model, the 30 dB-stronger interferer is rejected at the null angle.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(angles, res_steer, color="#1b9e77", linewidth=2, label="steer only (no null)")
ax.plot(angles, res_lcmv, color="#7570b3", linewidth=2, label="LCMV (null at -30)")
ax.axvline(20, color="#d95f02", linestyle=":", label="target +20 deg")
ax.axvline(-30, color="#7570b3", linestyle=":", label="interferer -30 deg")
ax.set_xlabel("Angle (deg)")
ax.set_ylabel("Response (dB)")
ax.set_title("Steering vs LCMV: the null changes everything at -30 deg")
ax.legend()
ax.set_ylim(-40, 2)
plt.tight_layout()
plt.show()

i30 = np.argmin(np.abs(angles + 30.0))
print(f"Null depth at -30 deg: {res_lcmv[i30] - res_steer[i30]:.1f} dB improvement over steering alone")
print(f"Target response: {res_steer[np.argmin(np.abs(angles-20))]:.1f} dB (steer) and {res_lcmv[np.argmin(np.abs(angles-20))]:.1f} dB (LCMV)")

## Checkpoint

In your own words, why does steering toward the target not remove a strong interferer that sits off to the side?

Then explain what the LCMV constraint C^H w = [1, 0] literally demands of the weights, and what that does to the beam at -30 degrees.


## Why this workflow nulls before the matched filter

The weights act on the raw array data, one array snapshot at each fast-time sample, before pulse compression in this implementation. The null uses direction information across the antennas; the matched filter uses echo delay. Combining first reduces the eight channels to one clean signal for subsequent range and Doppler processing. Because these are linear operations, matching all eight channels separately and then beamforming would give the same ideal result. What would fail is discarding the separate channels before their spatial phase pattern has been used.

Let us build the full array scene twice: once with only the target (a clean reference), and once with the interferer 30 dB stronger from -30 degrees. Both use the same noise stream, so the only difference is the interferer.


In [ ]:
pulse_len = int(round(radar_spec.pulse_width_s * radar_spec.fs_hz))
chirp = lfm_chirp(pulse_len, radar_spec.bandwidth_hz, radar_spec.pulse_width_s, radar_spec.fs_hz)
n_delay = delay_samples_for_range(radar_spec.target_range_m, radar_spec.fs_hz)

interferer_amp = 10.0 ** (30.0 / 20.0)  # 30 dB stronger than the target
v_target = 8.0
v_interferer = -8.0
fd_target = 2.0 * v_target / lam
fd_interferer = 2.0 * v_interferer / lam

a20 = steering_vector(np.radians(20.0), n_elements, d_spacing, lam)
am30 = steering_vector(np.radians(-30.0), n_elements, d_spacing, lam)

stack_target = build_pulse_stack(chirp, radar_spec.n_pulses, radar_spec.pri_s, fd_target, n_delay, amplitude=1.0)
stack_interf = build_pulse_stack(chirp, radar_spec.n_pulses, radar_spec.pri_s, fd_interferer, n_delay, amplitude=interferer_amp)

def build_scene(with_interferer):
    rng = np.random.default_rng(42)
    signals = np.zeros((n_elements, radar_spec.n_pulses, stack_target.shape[1]), dtype=complex)
    for n in range(n_elements):
        signals[n] = a20[n] * stack_target + (am30[n] * stack_interf if with_interferer else 0)
        signals[n] += np.sqrt(0.5) * (rng.standard_normal(signals[n].shape) + 1j * rng.standard_normal(signals[n].shape))
    return signals

element_clean = build_scene(False)  # target only: the reference
element_signals = build_scene(True)  # target plus strong interferer

print(f"element_signals shape: {element_signals.shape}  (elements x pulses x fast-time)")
print("Each element carries the target from +20 deg (and, in the second scene, the interferer from -30 deg) plus noise.")

## Range-Doppler before cancellation

First combine the elements with the *steering-only* weights (point at the target, no null) and then range-compress. The interferer leaks through the -18.6 dB sidelobe, so even though we aimed straight at the target, a loud return from the interferer's direction shows up in the same range bin and muddies the map.


In [ ]:
def beamform_rd(scene, w):
    combined = np.einsum('n,npm->pm', w.conj(), scene)
    profiles = np.array([matched_filter(combined[k], chirp) for k in range(radar_spec.n_pulses)])
    _, vel_axis, rdm = range_doppler_map(profiles, radar_spec.pri_s, radar_spec.fc_hz)
    return rdm, vel_axis

rdm_before, vel_before = beamform_rd(element_signals, w_steer)
range_axis = range_from_delay_samples(np.arange(rdm_before.shape[1]) - (pulse_len - 1), radar_spec.fs_hz)

def bin_value(rdm, vel_axis, vel_mps):
    r = np.argmin(np.abs(range_axis - radar_spec.target_range_m))
    return rdm[np.argmin(np.abs(vel_axis - vel_mps)), r]

# Control reference: target only, steered at the target - no interference at all.
rdm_control, vel_control = beamform_rd(element_clean, w_steer)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(rdm_before, aspect="auto", origin="lower",
               extent=[range_axis[0], range_axis[-1], vel_before[0], vel_before[-1]], cmap="viridis")
ax.axvline(radar_spec.target_range_m, color="white", linewidth=0.8, linestyle=":")
ax.set_xlim(700, 1300)
ax.set_ylim(-15, 15)
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("Range-Doppler before nulling (steer only)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Steering-only (no null), target + strong interferer:")
print(f"  target bin (+8 m/s)      = {bin_value(rdm_before, vel_before, v_target):.0f}")
print(f"  interferer bin (-8 m/s)  = {bin_value(rdm_before, vel_before, v_interferer):.0f}  (leaks through the sidelobe)")
print(f"\nReference: target only, no interferer -> target bin = {bin_value(rdm_control, vel_control, v_target):.0f}")

## Range-Doppler after cancellation

Now combine the elements with the LCMV weights - the same data, but this time the array itself nulls the interferer *before* the matched filter. The interferer's echo should vanish from its Doppler bin while the target's stays. This is the whole payoff of adaptive nulling: the interference is removed where the array can see it (in angle), so the range-Doppler processing downstream sees only the target.


In [ ]:
rdm_after, vel_after = beamform_rd(element_signals, w_lcmv)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(rdm_after, aspect="auto", origin="lower",
               extent=[range_axis[0], range_axis[-1], vel_after[0], vel_after[-1]], cmap="viridis")
ax.axvline(radar_spec.target_range_m, color="white", linewidth=0.8, linestyle=":")
ax.set_xlim(700, 1300)
ax.set_ylim(-15, 15)
ax.set_xlabel("Range (m)")
ax.set_ylabel("Velocity (m/s)")
ax.set_title("Range-Doppler after LCMV nulling")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("After LCMV nulling:")
print(f"  target bin (+8 m/s)      = {bin_value(rdm_after, vel_after, v_target):.0f}")
print(f"  interferer bin (-8 m/s)  = {bin_value(rdm_after, vel_after, v_interferer):.0f}  (nulled to the noise floor)")

## Before and after: the numbers

Compare the interferer's return before and after, side by side. Before, its Doppler-bin magnitude is about 75,692, versus about 20,743 in the target bin. After the null, the interferer bin drops to about 481, matching the clean-reference noise floor of about 480. The target bin remains near its clean-reference level of about 20,200. These values come from the fixed-seed model scene.


In [ ]:
print(f"target-only reference : target bin = {bin_value(rdm_control, vel_control, v_target):.0f}")
print(f"                       interferer bin = {bin_value(rdm_control, vel_control, v_interferer):.0f}  (noise floor)")
print()
print(f"{'':26}{'steer only':>12}{'LCMV null':>12}")
print(f"{"target bin (+8 m/s)":26}{bin_value(rdm_before, vel_before, v_target):>12.0f}{bin_value(rdm_after, vel_after, v_target):>12.0f}")
print(f"{"interferer bin (-8 m/s)":26}{bin_value(rdm_before, vel_before, v_interferer):>12.0f}{bin_value(rdm_after, vel_after, v_interferer):>12.0f}")
int_before = bin_value(rdm_before, vel_before, v_interferer)
int_after = bin_value(rdm_after, vel_after, v_interferer)
tgt_after = bin_value(rdm_after, vel_after, v_target)
tgt_ref = bin_value(rdm_control, vel_control, v_target)
print()
print(f"Interferer bin: {int_before:.0f} -> {int_after:.0f} (falls to the {bin_value(rdm_control, vel_control, v_interferer):.0f} noise floor)")
print(f"Target bin: {tgt_after:.0f} after nulling vs {tgt_ref:.0f} with no interferer - the target is preserved at its true level.")

## Common mistake

A common mistake is to think steering toward the target should be enough. Steering has no constraint to reject other directions, so it leaves sidelobes that a strong interferer can punch through. Here, 30 dB of input-power advantage minus 18.6 dB of sidelobe attenuation leaves roughly 11.4 dB at the beamformer output. Rejection requires an explicit null, which the constraint adds.

Another mistake is to collapse the eight antenna channels before applying spatial weights. The null uses the phase pattern across those channels. This notebook beamforms the raw array data first, then range-compresses the single output. Because both operations are linear, matching each antenna channel separately and then beamforming would also work if all eight channels remain available; beamforming a previously mixed single channel would not.


## Why the helpers exist

The cells above built the steering weights, solved the LCMV constraint, and applied the weights to the array data by hand so you can see that the whole trick is "choose w". Once the idea is clear, the same work collapses into steering_weights and lcmv_weights - so later notebooks or real experiments can point the array and null a jammer in a couple of lines.

Keep the first pass visible for the weight math, then use the helpers when the lesson moves on.


In [ ]:
# The same weights in helper form.
w_s = steering_weights(np.radians(20.0), n_elements, d_spacing, lam)
C_h, f_h, w_l = lcmv_weights(np.radians(20.0), [-np.radians(30.0)], n_elements, d_spacing, lam)

print(f"steering_weights -> main lobe at {angles[np.argmax(response_db(w_s))]:.0f} deg")
print(f"lcmv_weights     -> constraint matches {np.round(C_h.conj().T @ w_l, 4)}")

## Stretch: null a second interferer

The LCMV weight vector is not limited to one null - add whatever directions you need. Add a second, weaker interferer at +55 degrees and set up constraints for it too. Re-derive w to null both -30 and +55 degrees while keeping the target at 20. How many nulls can 8 elements support, and where would the design start to break down?


In [ ]:
# Two nulls: -30 deg and +55 deg, target still at +20 deg.
C2, f2, w2 = lcmv_weights(np.radians(20.0), [-np.radians(30.0), np.radians(55.0)], n_elements, d_spacing, lam)
res2 = response_db(w2)

print(f"Constraint check: {np.round(C2.conj().T @ w2, 4)}  (expect [1, 0, 0])")
print(f"Response at target  : {res2[np.argmin(np.abs(angles-20))]:.1f} dB")
print(f"Response at -30 deg : {res2[np.argmin(np.abs(angles+30))]:.1f} dB")
print(f"Response at +55 deg : {res2[np.argmin(np.abs(angles-55))]:.1f} dB")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(angles, res2, color="#d95f02", linewidth=2)
ax.axvline(20, color="#d95f02", linestyle=":", label="target +20 deg")
ax.axvline(-30, color="#7570b3", linestyle=":", label="null -30 deg")
ax.axvline(55, color="#1b9e77", linestyle=":", label="null +55 deg")
ax.set_ylim(-40, 2)
ax.legend()
ax.set_xlabel("Angle (deg)")
ax.set_ylabel("Response (dB)")
ax.set_title("LCMV with two nulls")
plt.tight_layout()
plt.show()

## Closing the loop: answers to the opening questions

At the start we listed five things to be able to explain. Here is each answer.

**What the array weights do and how steering points the lobe.** Every way of using the array reduces to one weight vector w, one complex gain per element. Steering chooses w = a(target)/N so a wave from the target's direction adds coherently and everything else falls into sidelobes; the main lobe points exactly at the target.

**How an LCMV constraint forces a null.** You collect the steering vectors of the target and interferer into C and demand C^H w = [1, 0] - unit response on the target, zero on the interferer. Solving w = C (C^H C)^{-1} f picks the weights that satisfy it while minimising output power.

**How deep the null is and why steering alone cannot give it.** The LCMV null at -30 degrees sat at about -319 dB, limited by numerical precision, versus the -18.6 dB sidelobe that steering alone left. Steering only points the main lobe; it has no constraint to reject other directions, so a loud source can punch through a sidelobe.

**Why the null happens before the matched filter.** The null is a spatial operation on the array snapshot - which angle a wave came from - while the matched filter is a temporal one - how far away it is. Cancelling in space first leaves a clean temporal signal to range- and Doppler-process.

**How the null removes the interferer from the range-Doppler map.** With steering-only weights the strong interferer fell at its -30 degree direction and inflated both Doppler bins at the target's range, so the map was muddled. After the LCMV null the interferer's own bin collapsed back to the noise floor while the target's bin matched the clean, interferer-free reference - the interferer is gone and the target is preserved.

If you can retell these five answers, you can point an array and deliberately silence a jammer while keeping the target - the sharpest tool in the beamforming kit.


## Summary

In this notebook you turned beamforming into a design choice. You steered the main lobe at the target with steering weights, then forced a deep null on the interferer with LCMV constraints C^H w = [1, 0]. The beam pattern showed the payoff: steering left a -18.6 dB sidelobe at the interferer, LCMV carved it down to about -319 dB while keeping the target at 0 dB.

You then applied the weights *before* the matched filter - the correct order, because the null is spatial and pulse compression is temporal. On the range-Doppler map, the 30 dB-stronger interferer's echo disappeared from its Doppler bin (falling back to the noise floor) while the target's echo matched its clean, interferer-free level. With range, velocity, direction, and now the ability to cancel interference by choice, you have the complete toolkit. The final notebook ties the whole chain together and assembles the portfolio artifacts.
